# Carnot HPR Comparison
This notebook compares direct and indirect Carnot-based heat pump and refrigeration targets on a real plant case and shows how the target load changes the utility picture.


## Case context
The packaged `chocolate_factory.json` case is a realistic multi-zone plant sample. This walkthrough keeps the workflow on the supported `PinchProblem.target_*` entry points, uses stable Carnot settings, and studies two target load levels so the load itself is treated as a decision variable.


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import plotly.graph_objects as go

from OpenPinch import PinchProblem
from OpenPinch.lib.enums import *
from OpenPinch.services.heat_pump_integration.common import (
    plot_multi_hp_profiles_from_results,
)
from OpenPinch.resources import copy_sample_case

PLOT_WIDTH = 720
PLOT_HEIGHT = 540


def display_plotly(fig, *, width: int = PLOT_WIDTH, height: int = PLOT_HEIGHT):
    fig = fig.to_dict()
    figure = go.Figure(fig)
    figure.update_layout(width=width, height=height, autosize=False)
    display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))

In [ ]:
workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
base_case_path = copy_sample_case(
    "chocolate_factory.json",
    work_dir / "chocolate_factory.json",
)
load_fractions = [0.25, 0.5, 0.75, 1.0]

In [ ]:
def _build_hpr_problem(load_fraction: float) -> PinchProblem:
    payload = json.loads(base_case_path.read_text(encoding="utf-8"))
    payload["options"] = {
        "HPR_TYPE": HPRcycle.MultiTempCarnot.value,
        "HPR_LOAD_VALUE": load_fraction,
        "MAX_HP_MULTISTART": 10,
        "N_COND": 3,
        "N_EVAP": 3,
        "REFRIGERANTS": "water;ammonia",
        "ALLOW_EXPANDER": True,
    }
    case_path = work_dir / f"chocolate_factory_hpr_{int(load_fraction * 100):02d}.json"
    case_path.write_text(json.dumps(payload), encoding="utf-8")
    return PinchProblem(problem_filepath=case_path)

In [ ]:
baseline_problem = _build_hpr_problem(load_fractions[-1])
hp = baseline_problem.target.direct_heat_pump()

In [ ]:
di = baseline_problem.master_zone.targets["Direct Heat Pump"]
print(baseline_problem.master_zone.targets)
print(di.pt)
print(hp.hpr_cop)
plot_multi_hp_profiles_from_results(
    T_hot=di.pt.col[ProblemTableLabel.T.value],
    T_cold=di.pt.col[ProblemTableLabel.T.value],
    H_hot=di.pt.col[ProblemTableLabel.H_NET_HOT.value],
    H_cold=di.pt.col[ProblemTableLabel.H_NET_COLD.value],
    hpr_hot_streams=hp.hpr_hot_streams,
    hpr_cold_streams=hp.hpr_cold_streams,
)

In [ ]:
baseline_summary = baseline_problem.summary_frame()
baseline_summary.loc[
    baseline_summary["Target"].isin(
        [
            "chocolate_factory_hpr_25/Direct Integration",
            "chocolate_factory_hpr_25/Total Process Target",
            "chocolate_factory_hpr_25/Total Site Target",
        ]
    ),
    ["Target", "Hot Utility Target", "Cold Utility Target", "Heat Recovery"],
].reset_index(drop=True)

In [ ]:
rows = []
for load_fraction in load_fractions:
    study_problem = _build_hpr_problem(load_fraction)
    study_problem.target()
    study_problem.target.direct_heat_pump(options={"HPR_LOAD_VALUE": load_fraction})
    study_problem.target.direct_refrigeration(options={"HPR_LOAD_VALUE": load_fraction})
    study_problem.target.indirect_heat_pump(options={"HPR_LOAD_VALUE": load_fraction})
    study_problem.target.indirect_refrigeration(
        options={"HPR_LOAD_VALUE": load_fraction}
    )

    for target in study_problem.results.targets:
        if "Heat Pump" not in target.name and "Refrigeration" not in target.name:
            continue
        rows.append(
            {
                "load fraction": load_fraction,
                "target": target.name.split("/", 1)[-1],
                "Qh": getattr(target.Qh, "value", target.Qh),
                "Qc": getattr(target.Qc, "value", target.Qc),
                "Qr": getattr(target.Qr, "value", target.Qr),
                "hpr_work": target.hpr_work,
                "hpr_cop": target.hpr_cop,
                "hpr_success": target.hpr_success,
            }
        )

comparison = pd.DataFrame(rows)
comparison

In [ ]:
reference_problem = _build_hpr_problem(load_fractions[0])
reference_problem.target()
reference_problem.target.direct_heat_pump(options={"HPR_LOAD_VALUE": load_fractions[0]})
reference_problem.target.indirect_heat_pump(
    options={"HPR_LOAD_VALUE": load_fractions[0]}
)
reference_problem.target.indirect_refrigeration(
    options={"HPR_LOAD_VALUE": load_fractions[0]}
)
{
    "available_hpr_graphs": sorted(
        reference_problem.graph_catalog()
        .loc[
            reference_problem.graph_catalog()["Graph Type"].str.contains(
                "Heat Pump|Site Utility",
                regex=True,
            ),
            "Graph Type",
        ]
        .unique()
    ),
    "direct_hp_graph_traces": len(
        reference_problem.plot(graph_type="Grand Composite Curve with Heat Pump").data
    ),
    "site_graph_traces": len(
        reference_problem.plot(graph_type="Site Utility Grand Composite Curve").data
    ),
}

## Inspect these outputs first
Start with the baseline direct-integration and total-site rows so you know the untreated utility picture. Then read the comparison table by load fraction: direct versus indirect placement is the first comparison, and heat pump versus refrigeration is the second. The supporting HPR and site-utility graphs help explain why the utility targets moved.

## Interpretation
A useful HPR option should improve the utility picture at a load that is still operationally plausible. If the better option changes with load fraction, the load target is part of the decision and should be documented explicitly rather than treated as a hidden model setting.
